In [2]:
                                                    # Real time Hand Gesture Prediction Script

import cv2
import numpy as np
import tensorflow as tf

# -------- CONFIG --------
MODEL_PATH = "efficientnet_finetuned_final&final.h5"  # path to your fine-tuned model
IMG_SIZE = (300, 300)                     # input size of your model

# Your actual class names in correct order
CLASS_NAMES = [
    "down", "eight", "five", "four", "left",
    "nine", "one", "right", "seven", "six",
    "stop", "three", "two", "up", "zero"
]
# ------------------------

# Load the fine-tuned model
model = tf.keras.models.load_model(MODEL_PATH)

# Open webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("❌ Cannot access webcam")
    exit()

print("🎯 Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    # Draw guide rectangle (center 300x300)
    h, w, _ = frame.shape
    x1, y1 = w//2 - 150, h//2 - 150
    x2, y2 = w//2 + 150, h//2 + 150
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Crop ROI from center
    roi = frame[y1:y2, x1:x2]
    roi_resized = cv2.resize(roi, IMG_SIZE)
    roi_normalized = roi_resized / 255.0
    roi_input = np.expand_dims(roi_normalized, axis=0)

    # Make prediction
    preds = model.predict(roi_input, verbose=0)
    class_idx = np.argmax(preds[0])
    class_label = CLASS_NAMES[class_idx]
    confidence = preds[0][class_idx]

    # Display prediction on frame
    cv2.putText(frame,
                f"{class_label} ",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2)

    cv2.imshow("Hand Gesture Prediction", frame)

    # Quit on pressing 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


🎯 Press 'q' to quit
